In [96]:
!python --version

Python 3.11.9


In [97]:
%matplotlib inline

In [98]:
import mne
mne.set_log_level("ERROR")

In [99]:

# EEG Config
EEG_TMIN = 0.0
EEG_TMAX = 2.0

ERP_WINDOW_START = 1.1
ERP_WINDOW_END = 1.4


# fNIRS Config
FNIRS_CHANNELS = [
    "fnirs_1",
    "fnirs_2",
    "fnirs_3",
    "fnirs_4",
    "fnirs_5",
    "fnirs_6",
    "fnirs_7",
    "fnirs_8"
]
FNIRS_FS=64
# Bandpass filter
FNIRS_LOW = 0.01
FNIRS_HIGH = 0.3

# Hemodynamic response window
FNIRS_TMIN = 2.0
FNIRS_TMAX = 8.0

SHR_WINDOW_START = 4.5
SHR_WINDOW_END = 5.5

In [100]:
import os

def define_auth():
    os.environ["HF_TOKEN"] = ""

In [101]:
import pandas as pd

def get_dataset(type="train"):
    splits = {'train': 'train_meta.csv', 'test': 'test_meta.csv'}
    df = pd.read_csv("hf://datasets/lasfk/EEG-fNIRS-based-Handwriting-Trajectory-Dataset/" + splits[type])
    return df

## EEG Extraction

In [102]:
import mne

def read_bdf(subject, session):
    path = f"datasets/raw/{subject}/EEG/{session}.bdf"
    raw = mne.io.read_raw_bdf(path, preload=True)
    return raw

In [103]:
import numpy as np

from asrpy import ASR

def clean_bdf(raw):
    empty_channels = [
    'Fpz',
    'Fp1',
    'Fp2',
    'AF3',
    'AF4',
    'AF7',
    'AF8',
    'F7',
    'F8',
    'FT7',
    'FT8',
    'T7',
    'T8',
    'TP7',
    'TP8',
    'Pz',
    'P3',
    'P4',
    'P5',
    'P6',
    'P7',
    'P8',
    'POz',
    'PO3',
    'PO4',
    'PO5',
    'PO6',
    'PO7',
    'PO8',
    'Oz',
    'O1',
    'O2',
    'ECG',
    'HEOR',
    'HEOL',
    'VEOU',
    'VEOL',
    ]
    raw.drop_channels(empty_channels)

def segmentation_eeg(
    raw,
    onset_sec,
    label,
    tmin,
    tmax
):
    sfreq = raw.info["sfreq"]

    start_sample = int((onset_sec + tmin) * sfreq)
    end_sample = int((onset_sec + tmax) * sfreq)

    if (start_sample < 0 or end_sample > raw.n_times):
        return None, -1


    eeg_data = raw.get_data()

    segment = eeg_data[:,start_sample:end_sample]

    return segment, label

def normalize_channel(X):
    X_norm = np.zeros_like(X)

    for i in range(X.shape[0]):
    
        for ch in range(X.shape[1]):
    
            signal = X[i, ch]
    
            mean = signal.mean()
            std = signal.std()
    
            if std == 0:
                std = 1e-8
    
            X_norm[i, ch] = (signal - mean) / std
    return X_norm

def asr_remove(raw):
    sfreq = raw.info["sfreq"]
    asr = ASR(
        sfreq=sfreq,
        cutoff=20
    )
    asr.fit(raw)
    clean_data = asr.transform(raw)
    return mne.io.RawArray(clean_data, raw.info)

def preprocessing_bdf(raw):
    clean_bdf(raw)

    # Buang sinyal listrik
    raw.notch_filter(50)

    # Band Pass 1Hz~45Hz, Band EEG umum => Delta, Theta, Alpha, Beta, Gamma (1 - 45)Hz
    raw.filter(0.5, 45)
    return raw

## fNIRS Extraction

In [104]:
import pandas as pd

def read_fnirs(subject, session):
    path = f"datasets/raw/{subject}/fNIRS/{session}.csv"
    fnirs = pd.read_csv(path)
    return fnirs

In [105]:
import numpy as np

from scipy.signal import butter
from scipy.stats import median_abs_deviation
from sklearn.decomposition import PCA

from scipy.signal import butter
from scipy.signal import sosfiltfilt


def segmentation_fnirs(
    df_fnirs,
    onset_sec,
    label,
    tmin,
    tmax,
    fs
):
    start_sample = int((onset_sec + tmin) * fs)
    end_sample = int((onset_sec + tmax) * fs)
    fnirs_data = df_fnirs
    if (start_sample < 0 or end_sample > fnirs_data.shape[1]):
        return None, -1

    segment = fnirs_data[:, start_sample:end_sample]

    return segment, label

def bandpass_fnirs(
    signal,
    fs=FNIRS_FS,
    lowcut=FNIRS_LOW,
    highcut=FNIRS_HIGH,
    order=4
):

    signal = np.nan_to_num(
        signal,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    nyquist = fs / 2

    low = lowcut / nyquist
    high = highcut / nyquist

    sos = butter(
        order,
        [low, high],
        btype="bandpass",
        output="sos"
    )

    filtered = sosfiltfilt(
        sos,
        signal
    )

    filtered = np.nan_to_num(
        filtered,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return filtered

def robust_clip(
    signal,
    threshold=5.0
):

    median = np.median(signal)

    mad = median_abs_deviation(
        signal
    )

    if mad == 0:
        mad = 1e-8

    robust_z = (
        signal - median
    ) / mad

    clipped = signal.copy()

    clipped[
        np.abs(robust_z) > threshold
    ] = median

    return clipped


def remove_global_pca(
    fnirs_data
):
    pca = PCA(
        n_components=fnirs_data.shape[0]
    )

    transformed = pca.fit_transform(
        fnirs_data.T
    )

    transformed[:, 0] = 0

    reconstructed = pca.inverse_transform(
        transformed
    ).T

    return reconstructed

def normalize_fnirs(
    fnirs_data
):

    normalized = np.zeros_like(
        fnirs_data
    )

    for ch in range(fnirs_data.shape[0]):

        signal = fnirs_data[ch]

        mean = signal.mean()
        std = signal.std()

        if std == 0:
            std = 1e-8

        normalized[ch] = (
            signal - mean
        ) / std

    return normalized


def preprocessing_fnirs(fnirs_data, fs):

    # =====================================================
    # DATAFRAME -> NUMPY
    # =====================================================

    if isinstance(fnirs_data, pd.DataFrame):

        fnirs_data = fnirs_data.values.T

    # shape:
    # (channels, times)

    processed = fnirs_data.copy()

    # =====================================================
    # BANDPASS
    # =====================================================

    for ch in range(processed.shape[0]):

        signal = processed[ch]

        signal = bandpass_fnirs(
            signal,
            fs=fs,
            lowcut=0.01,
            highcut=0.3
        )

        processed[ch] = signal

    # =====================================================
    # PCA
    # =====================================================

    pca = PCA(
        n_components=processed.shape[0]
    )

    transformed = pca.fit_transform(
        processed.T
    )

    # -----------------------------------------------------
    # REMOVE PC1
    # -----------------------------------------------------

    transformed[:, 0] = 0

    # =====================================================
    # RECONSTRUCT
    # =====================================================

    reconstructed = pca.inverse_transform(
        transformed
    )

    # shape:
    # (times, channels)

    reconstructed = reconstructed.T

    return reconstructed

## Extract Features

In [106]:
define_auth()
df_train = get_dataset(type="train")

print(df_train.head())

      trial_id subject  session  onset_sec  event_code  label
0  sub_01_1_00  sub_01        1     25.060         203      3
1  sub_01_1_01  sub_01        1     49.410         201      1
2  sub_01_1_02  sub_01        1     73.870         201      1
3  sub_01_1_03  sub_01        1     98.695         201      1
4  sub_01_1_04  sub_01        1    123.850         202      2


### Perbedaan trial onset_sec diff time

In [107]:
df_train["next_onset"] = (
    df_train["onset_sec"]
    .shift(-1)
)

df_train["diff"] = (
    df_train["next_onset"]
    -
    df_train["onset_sec"]
)

print(df_train["diff"].describe())

count    6442.000000
mean        0.137596
std       151.903929
min     -1345.378000
25%        24.852000
50%        24.858000
75%        24.864000
max       366.888000
Name: diff, dtype: float64


### Extract Features EEG & fNIRS

In [108]:
import numpy as np

from tqdm import tqdm


subject_erp_features = {}
subject_shr_features = {}



subjects = sorted(
    df_train["subject"].unique()
)

for subject in tqdm(subjects):
    print(f"\nProcessing Subject {subject}")

    subject_df = df_train[
        df_train["subject"] == subject
    ]

    eeg_trials = []
    eeg_labels = []

    
    fnirs_trials = []
    fnirs_labels = []
    
    sessions = sorted(
        subject_df["session"].unique()
    )

    for session in sessions:
        print(f"\nProcessing Session {session}")
        raw = read_bdf(
            subject,
            session
        )

        raw = preprocessing_bdf(
            raw
        )


        df_fnirs = read_fnirs(
            subject,
            session
        )

        df_fnirs = preprocessing_fnirs(
            df_fnirs,
            FNIRS_FS,
        )

    
        session_df = subject_df[
            subject_df["session"] == session
        ]

        for _, row in session_df.iterrows():

            onset_sec = row["onset_sec"]

            label = int(
                row["label"]
            )

            # =============================================
            # EEG SEGMENT
            # =============================================

            segment_eeg, _ = segmentation_eeg(
                raw,
                onset_sec,
                label,
                EEG_TMIN,
                EEG_TMAX
            )

            # =============================================
            # FNIRS SEGMENT
            # =============================================

            segment_fnirs, _ = segmentation_fnirs(
                df_fnirs,
                onset_sec,
                label,
                FNIRS_TMIN,
                FNIRS_TMAX,
                FNIRS_FS
            )

            # =====================================================
            # VALIDATE EEG SHAPE
            # =====================================================
            
            EXPECTED_EEG_SAMPLES = int(
                (EEG_TMAX - EEG_TMIN)
                *
                raw.info["sfreq"]
            )
            
            if (
                segment_eeg is None
                or
                segment_eeg.shape[1]
                !=
                EXPECTED_EEG_SAMPLES
            ):
                continue
            
            # =====================================================
            # VALIDATE FNIRS SHAPE
            # =====================================================
            
            EXPECTED_FNIRS_SAMPLES = int(
                (FNIRS_TMAX - FNIRS_TMIN)
                *
                FNIRS_FS
            )
            
            if (
                segment_fnirs is None
                or
                segment_fnirs.shape[1]
                !=
                EXPECTED_FNIRS_SAMPLES
            ):
                continue

            # =============================================
            # SAVE
            # =============================================

            eeg_trials.append(
                segment_eeg
            )

            eeg_labels.append(
                label
            )

            fnirs_trials.append(
                segment_fnirs
            )

            fnirs_labels.append(
                label
            )

    # =====================================================
    # ARRAY
    # =====================================================

    eeg_trials = np.array(
        eeg_trials
    )

    eeg_labels = np.array(
        eeg_labels
    )

    fnirs_trials = np.array(
        fnirs_trials
    )

    fnirs_labels = np.array(
        fnirs_labels
    )

    # =====================================================
    # REMOVE NaN
    # =====================================================

    eeg_trials = np.nan_to_num(
        eeg_trials
    )

    fnirs_trials = np.nan_to_num(
        fnirs_trials
    )

    # =====================================================
    # NORMALIZE
    # =====================================================

    eeg_trials = normalize_channel(
        eeg_trials
    )

    fnirs_trials = normalize_fnirs(
        fnirs_trials
    )

    # =====================================================
    # EEG TIME
    # =====================================================

    eeg_times = np.linspace(
        EEG_TMIN,
        EEG_TMAX,
        eeg_trials.shape[-1]
    )

    eeg_mask = (
        (eeg_times >= ERP_WINDOW_START)
        &
        (eeg_times <= ERP_WINDOW_END)
    )

    # =====================================================
    # FNIRS TIME
    # =====================================================

    fnirs_times = np.linspace(
        FNIRS_TMIN,
        FNIRS_TMAX,
        fnirs_trials.shape[-1]
    )

    fnirs_mask = (
        (fnirs_times >= SHR_WINDOW_START)
        &
        (fnirs_times <= SHR_WINDOW_END)
    )

    # =====================================================
    # ERP FEATURE PER CLASS
    # =====================================================

    erp_feature_per_class = {}

    for cls in range(4):

        class_trials = eeg_trials[
            eeg_labels == cls
        ]

        # -------------------------------------------------
        # ERP
        # -------------------------------------------------

        erp = class_trials.mean(
            axis=0
        )

        # shape:
        # (channels, times)

        # -------------------------------------------------
        # WINDOW
        # -------------------------------------------------

        erp_window = erp[
            :,
            eeg_mask
        ]

        # -------------------------------------------------
        # FEATURE
        # -------------------------------------------------

        mean_amp = erp_window.mean(
            axis=1
        )

        peak = erp_window.max(
            axis=1
        )

        auc = np.trapezoid(
            erp_window,
            axis=1
        )

        feat = np.concatenate([
            mean_amp,
            peak,
            auc
        ])

        erp_feature_per_class[
            cls
        ] = feat

    # =====================================================
    # SHR FEATURE PER CLASS
    # =====================================================

    shr_feature_per_class = {}

    for cls in range(4):

        class_trials = fnirs_trials[
            fnirs_labels == cls
        ]

        # -------------------------------------------------
        # SHR
        # -------------------------------------------------

        shr = class_trials.mean(
            axis=0
        )

        # shape:
        # (channels, times)

        # -------------------------------------------------
        # WINDOW
        # -------------------------------------------------

        shr_window = shr[
            :,
            fnirs_mask
        ]

        # -------------------------------------------------
        # FEATURE
        # -------------------------------------------------

        mean_amp = shr_window.mean(
            axis=1
        )

        peak = shr_window.max(
            axis=1
        )

        auc = np.trapezoid(
            shr_window,
            axis=1
        )

        slope = (
            shr_window[:, -1]
            -
            shr_window[:, 0]
        )

        feat = np.concatenate([
            mean_amp,
            peak,
            auc,
            slope
        ])

        shr_feature_per_class[
            cls
        ] = feat

    # =====================================================
    # SAVE SUBJECT FEATURE
    # =====================================================

    subject_erp_features[
        subject
    ] = erp_feature_per_class

    subject_shr_features[
        subject
    ] = shr_feature_per_class

# =========================================================
# DONE
# =========================================================

print("\nDONE")

print(
    "\nERP SUBJECT COUNT:",
    len(subject_erp_features)
)

print(
    "SHR SUBJECT COUNT:",
    len(subject_shr_features)
)

# =========================================================
# EXAMPLE
# =========================================================

example_subject = subjects[0]

print(
    "\nExample ERP Feature Shape:"
)

print(
    subject_erp_features[
        example_subject
    ][0].shape
)

print(
    "\nExample SHR Feature Shape:"
)

print(
    subject_shr_features[
        example_subject
    ][0].shape
)

  0%|                                                                                                                                                                                                      | 0/20 [00:00<?, ?it/s]


Processing Subject sub_01


  5%|█████████▌                                                                                                                                                                                    | 1/20 [01:08<21:45, 68.71s/it]


Processing Subject sub_02


 10%|███████████████████                                                                                                                                                                           | 2/20 [02:16<20:26, 68.11s/it]

...


Processing Subject sub_19


 95%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌         | 19/20 [20:53<01:06, 66.25s/it]


Processing Subject sub_20


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [21:52<00:00, 65.63s/it]


DONE

ERP SUBJECT COUNT: 20
SHR SUBJECT COUNT: 20

Example ERP Feature Shape:
(81,)

Example SHR Feature Shape:
(36,)


In [111]:
X_feature = []

y_feature = []

subject_ids = []

# =====================================================
# SUBJECT LOOP
# =====================================================

for subject in subject_erp_features.keys():

    # -------------------------------------------------
    # CLASS LOOP
    # -------------------------------------------------

    for cls in range(4):

        # =============================================
        # ERP FEATURE
        # =============================================

        erp_feat = subject_erp_features[
            subject
        ][cls]

        # =============================================
        # SHR FEATURE
        # =============================================

        shr_feat = subject_shr_features[
            subject
        ][cls]

        # =============================================
        # CONCAT MULTIMODAL
        # =============================================

        multimodal_feat = np.concatenate([
            erp_feat,
            shr_feat
        ])

        # =============================================
        # SAVE
        # =============================================

        X_feature.append(
            multimodal_feat
        )

        y_feature.append(
            cls
        )

        subject_ids.append(
            subject
        )

# =====================================================
# ARRAY
# =====================================================

X_feature = np.array(
    X_feature
)

y_feature = np.array(
    y_feature
)

subject_ids = np.array(
    subject_ids
)

# =====================================================
# SHAPE
# =====================================================

print("\nX_feature Shape")
print(
    X_feature.shape
)

print("\ny_feature Shape")
print(
    y_feature.shape
)

print("\nsubject_ids Shape")
print(
    subject_ids.shape
)


X_feature Shape
(80, 117)

y_feature Shape
(80,)

subject_ids Shape
(80,)


## Training Model

### Check Device

In [112]:
import torch

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(DEVICE)

True
NVIDIA GeForce RTX 4070
cuda


In [115]:
logo = LeaveOneGroupOut()

accuracies = []

for fold, (train_idx,test_idx) in enumerate(

    logo.split(
        X_feature,
        y_feature,
        groups=subject_ids
    )

):

    print(
        f"\n========== FOLD {fold+1} =========="
    )

    # =====================================================
    # SPLIT
    # =====================================================

    X_train = X_feature[
        train_idx
    ]

    X_test = X_feature[
        test_idx
    ]

    y_train = y_feature[
        train_idx
    ]

    y_test = y_feature[
        test_idx
    ]

    # =====================================================
    # STANDARDIZE
    # =====================================================

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        X_train
    )

    X_test = scaler.transform(
        X_test
    )

    # =====================================================
    # SVM
    # =====================================================

    svm = SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale"
    )

    svm.fit(
        X_train,
        y_train
    )

    # =====================================================
    # PREDICT
    # =====================================================

    y_pred = svm.predict(
        X_test
    )

    # =====================================================
    # ACCURACY
    # =====================================================

    acc = accuracy_score(
        y_test,
        y_pred
    )

    accuracies.append(
        acc
    )

    test_subject = np.unique(
        subject_ids[test_idx]
    )

    print(
        "Test Subject:",
        test_subject
    )

    print(
        "Accuracy:",
        round(acc, 4)
    )

# =========================================================
# FINAL RESULT
# =========================================================

print("\n==============================")
print("FINAL LOSO RESULT")
print("==============================")

print(
    "Mean Accuracy:",
    np.mean(accuracies)
)

print(
    "Std Accuracy:",
    np.std(accuracies)
)


========== FOLD 1 ==========
Test Subject: ['sub_01']
Accuracy: 0.25

========== FOLD 2 ==========
Test Subject: ['sub_02']
Accuracy: 0.25

========== FOLD 3 ==========
Test Subject: ['sub_03']
Accuracy: 0.75

========== FOLD 4 ==========
Test Subject: ['sub_04']
Accuracy: 0.5

========== FOLD 5 ==========
Test Subject: ['sub_05']
Accuracy: 0.25

========== FOLD 6 ==========
Test Subject: ['sub_06']
Accuracy: 0.5

========== FOLD 7 ==========
Test Subject: ['sub_07']
Accuracy: 0.0

========== FOLD 8 ==========
Test Subject: ['sub_08']
Accuracy: 0.5

========== FOLD 9 ==========
Test Subject: ['sub_09']
Accuracy: 0.5

========== FOLD 10 ==========
Test Subject: ['sub_10']
Accuracy: 0.25

========== FOLD 11 ==========
Test Subject: ['sub_11']
Accuracy: 0.25

========== FOLD 12 ==========
Test Subject: ['sub_12']
Accuracy: 0.25

========== FOLD 13 ==========
Test Subject: ['sub_13']
Accuracy: 0.25

========== FOLD 14 ==========
Test Subject: ['sub_14']
Accuracy: 0.25

========== FOLD 15 

In [127]:
import numpy as np

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.model_selection import (
    train_test_split
)

from sklearn.preprocessing import (
    StandardScaler
)

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_feature,
    y_feature,
    test_size=0.2,
    random_state=42,
    stratify=y_feature
)

print("\nTrain Shape")
print(X_train.shape)

print("\nTest Shape")
print(X_test.shape)

# =========================================================
# STANDARDIZE
# =========================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
)

X_test = scaler.transform(
    X_test
)

# =========================================================
# SVM
# =========================================================

svm = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale"
)

# =========================================================
# TRAIN
# =========================================================

svm.fit(
    X_train,
    y_train
)

# =========================================================
# PREDICT
# =========================================================

y_pred = svm.predict(
    X_test
)

# =========================================================
# ACCURACY
# =========================================================

acc = accuracy_score(
    y_test,
    y_pred
)

print("\n==============================")
print("SVM RESULT")
print("==============================")

print(
    "Accuracy:",
    round(acc, 4)
)

# =========================================================
# CLASSIFICATION REPORT
# =========================================================

print("\nClassification Report")

print(
    classification_report(
        y_test,
        y_pred
    )
)

# =========================================================
# CONFUSION MATRIX
# =========================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix")

print(cm)


Train Shape
(64, 117)

Test Shape
(16, 117)

SVM RESULT
Accuracy: 0.3125

Classification Report
              precision    recall  f1-score   support

           0       0.33      0.25      0.29         4
           1       0.50      0.50      0.50         4
           2       0.29      0.50      0.36         4
           3       0.00      0.00      0.00         4

    accuracy                           0.31        16
   macro avg       0.28      0.31      0.29        16
weighted avg       0.28      0.31      0.29        16


Confusion Matrix
[[1 0 2 1]
 [0 2 2 0]
 [0 1 2 1]
 [2 1 1 0]]


## Submission

In [53]:
import pandas as pd

df_test = get_dataset(type="test")
print(df_test.head())

      trial_id subject  session  onset_sec
0  sub_21_1_00  sub_21        1     24.495
1  sub_21_1_01  sub_21        1     49.347
2  sub_21_1_02  sub_21        1     74.205
3  sub_21_1_03  sub_21        1     99.075
4  sub_21_1_04  sub_21        1    123.933


In [54]:
import mne

mne.set_log_level("ERROR")
last_subject=""
last_session=0
raw=None
df_fnirs=None

EEG_TMIN = 0.0
EEG_TMAX = 4.0

# Dari explanatory data analysis
FNIRS_FS = 64

# 0.01 Hz ≤ f ≤ 0.2 Hz
# Kita hanya ambil perubahan sinyal yang:
# tidak terlalu lambat
# tidak terlalu cepat
FNIRS_LOW = 0.01
FNIRS_HIGH = 0.2

# Dimulai + 2 Second = menunggu respons aliran darah mulai muncul.
FNIRS_TMIN = 2.0
FNIRS_TMAX = 8.0

X_test_eeg = []
X_test_fnirs = []
trial_ids = []
subject_ids = []

for _, row in tqdm(
    df_test.iterrows(),
    total=len(df_test)
):
    subject = row["subject"]
    session = row["session"]
    onset_sec = row["onset_sec"]
    trial_id = row["trial_id"]

    if last_subject != subject or last_session != session:
        # EEG
        raw = read_bdf(subject, session)
        raw = preprocessing_bdf(raw)

        # fNIRS
        df_fnirs = read_fnirs(subject, session)
        df_fnirs = preprocessing_fnirs(df_fnirs, FNIRS_FS, FNIRS_LOW, FNIRS_HIGH)
        
        last_subject = subject
        last_session = session
        
    # Segmentation BDF
    segment_eeg, label_eeg = segmentation_eeg(raw, onset_sec, label, EEG_TMIN, EEG_TMAX)
    
    # Segmentation fNIRS
    segment_fnirs, label_fnirs = segmentation_fnirs(
        df_fnirs,
        onset_sec,
        label,
        FNIRS_TMIN,
        FNIRS_TMAX,
        FNIRS_FS
    )
    if label_eeg == -1 or label_fnirs == -1:
        continue

    X_test_eeg.append(
        segment_eeg
    )

    X_test_fnirs.append(
        segment_fnirs
    )

    trial_ids.append(
        trial_id
    )
    subject_ids.append(subject)

X_test_eeg = np.array(X_test_eeg)
X_test_fnirs = np.array(X_test_fnirs)
subject_ids = np.array(subject_ids)
X_test_eeg = np.nan_to_num(
    X_test_eeg,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

X_test_fnirs = np.nan_to_num(
    X_test_fnirs,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

# Normalize EEG
X_test_eeg = normalize_channel(X_test_eeg)

# Normalize fNIRS
X_test_fnirs = normalize_fnirs(X_test_fnirs)

print("Test EEG shape:")
print(X_test_eeg.shape)

print("Test fNIRS shape:")
print(X_test_fnirs.shape)


100%|██████████████████████████████████████████████████████████████████████████████| 3238/3238 [03:20<00:00, 16.18it/s]


Test EEG shape:
(3205, 27, 4000)
Test fNIRS shape:
(3205, 8, 384)


In [55]:
# =========================================================
# TEST DATASET
# =========================================================

class TestDataset(Dataset):
    def __init__(self, eeg, fnirs):
        self.eeg = eeg
        self.fnirs = fnirs

    def __len__(self):
        return len(self.eeg)

    def __getitem__(self, idx):
        eeg = torch.tensor(self.eeg[idx], dtype=torch.float32)
        fnirs = torch.tensor(self.fnirs[idx], dtype=torch.float32)

        return eeg, fnirs


test_dataset = TestDataset(X_test_eeg, X_test_fnirs)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# =========================================================
# LOAD BEST MODEL
# =========================================================

eeg_channels = X_test_eeg.shape[1]
fnirs_channels = X_test_fnirs.shape[1]
model = MultiModalNet(
    eeg_channels,
    fnirs_channels,
    NUM_CLASSES
).to(DEVICE)

# pakai fold terbaikmu
BEST_FOLD = 0

model.load_state_dict(
    torch.load(f"best_fold_{BEST_FOLD}.pth", weights_only=True)
)

model.eval()


# =========================================================
# INFERENCE
# =========================================================

predictions = []

with torch.no_grad():
    for eeg, fnirs in tqdm(test_loader):
        eeg = eeg.to(DEVICE)
        fnirs = fnirs.to(DEVICE)
        
        with torch.amp.autocast("cuda"):
            outputs = model(eeg, fnirs)
            
        preds = torch.argmax(outputs, dim=1)
        predictions.extend(preds.cpu().numpy())


predictions = np.array(predictions)

print("Predictions shape:")
print(predictions.shape)


# =========================================================
# CREATE SUBMISSION
# =========================================================

submission = pd.DataFrame({
    "trial_id": trial_ids,
    "label": predictions
})

print(submission.head())


# =========================================================
# SAVE CSV
# =========================================================

submission.to_csv(
    "submission.csv",
    index=False
)

print("\nsubmission.csv saved!")

100%|███████████████████████████████████████████████████████████████████████████████| 101/101 [00:00<00:00, 109.88it/s]


Predictions shape:
(3205,)
      trial_id  label
0  sub_21_1_00      2
1  sub_21_1_01      3
2  sub_21_1_02      3
3  sub_21_1_03      3
4  sub_21_1_04      2

submission.csv saved!
